# 4.4 Persistence and testing

Convert CSV rows into objects and processed objects back into CSV rows.

In [ ]:
from pathlib import Path
import csv
import tempfile

def required(value, label):
    cleaned = value.strip()
    if not cleaned:
        raise ValueError(f"{label} must not be empty")
    return cleaned

class EquipmentItem:
    def __init__(self, item_id, name, category, borrower_id=""):
        self.item_id = required(item_id, "item_id")
        self.name = required(name, "name")
        self.category = required(category, "category")
        self.borrower_id = borrower_id.strip() or None

    def is_available(self):
        return self.borrower_id is None

    def loan_to(self, borrower_id):
        borrower_id = required(borrower_id, "borrower_id")
        if not self.is_available():
            raise ValueError("item is already on loan")
        self.borrower_id = borrower_id

    def return_item(self):
        if self.is_available():
            raise ValueError("item is not on loan")
        self.borrower_id = None

    def to_record(self):
        return {"item_id": self.item_id, "name": self.name,
                "category": self.category, "borrower_id": self.borrower_id or ""}

class LendingDesk:
    def __init__(self):
        self.items = {}

    def add_item(self, item):
        if not isinstance(item, EquipmentItem):
            raise TypeError("item must be EquipmentItem")
        if item.item_id in self.items:
            raise ValueError("duplicate item ID")
        self.items[item.item_id] = item

    def find_item(self, item_id):
        return self.items.get(item_id.strip())

    def required_item(self, item_id):
        item = self.find_item(item_id)
        if item is None:
            raise KeyError(item_id)
        return item

    def loan_item(self, item_id, borrower_id):
        self.required_item(item_id).loan_to(borrower_id)

    def return_item(self, item_id):
        self.required_item(item_id).return_item()

    def summary(self):
        available = sum(x.is_available() for x in self.items.values())
        return {"total_items": len(self.items), "available_items": available,
                "loaned_items": len(self.items) - available}

## Round-trip one record

In [ ]:
row = {"item_id": "X1", "name": "Tablet", "category": "Computer", "borrower_id": ""}
obj = EquipmentItem(row["item_id"], row["name"], row["category"], row["borrower_id"])
assert obj.to_record() == row
print(obj.to_record())

## Load a temporary CSV

Different data shows that code does not hard-code published IDs.

In [ ]:
def load_inventory(path):
    result = LendingDesk()
    with Path(path).open(newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            result.add_item(EquipmentItem(row["item_id"], row["name"], row["category"], row["borrower_id"]))
    return result

folder = tempfile.TemporaryDirectory()
source = Path(folder.name) / "inventory.csv"
source.write_text("item_id,name,category,borrower_id\nX1,Tablet,Computer,\nX2,Camera,Media,U9\n", encoding="utf-8")
loaded = load_inventory(source)
print(loaded.summary())

## Save without changing the source

In [ ]:
def save_inventory(counter, path):
    path = Path(path)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["item_id","name","category","borrower_id"])
        writer.writeheader()
        for item_id in sorted(counter.items):
            writer.writerow(counter.items[item_id].to_record())

before = source.read_bytes()
output = Path(folder.name) / "inventory_after.csv"
save_inventory(loaded, output)
assert source.read_bytes() == before
print(output.read_text(encoding="utf-8"))

## Turn expected rejection into an audit result

Catch published ValueError and KeyError as REJECTED. Do not hide unexpected defects.

In [ ]:
requests = [("Q1","LOAN","X1","U1"), ("Q2","LOAN","X2","U2"), ("Q3","RETURN","BAD","")]
results = []
for request_id, action, item_id, borrower in requests:
    try:
        if action == "LOAN":
            loaded.loan_item(item_id, borrower)
        else:
            loaded.return_item(item_id)
        status = "ACCEPTED"
    except (ValueError, KeyError):
        status = "REJECTED"
    results.append((request_id, status))
print(results)

## Integrated check

Assert accepted + rejected = requests, unchanged source, reloadable output, and preserved state after rejection.